<a href="https://colab.research.google.com/github/TheSarthakShinde/GENAI-/blob/main/genaiexp9rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("Enable a GPU runtime before continuing.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
%pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    peft \
    trl \
    datasets \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    pandas \
    scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 84.8 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
def generate_answer(prompt, max_new_tokens=250):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful AI assistant. "
                "Answer accurately and clearly. "
                "If you do not know something, say so."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[
        :, inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True
    )

In [ ]:
print(
    generate_answer(
        "Explain Retrieval-Augmented Generation in simple English."
    )
)

Retrieval-Augmented Generation is a type of language model that combines two main parts: a retrieval system and a generation system.

1. **Retrieval System**: This part of the model is like a big library where it stores information from various sources such as books, articles, or web pages. When asked a question or given some context, this system can quickly find relevant pieces of information stored in its library.

2. **Generation System**: This part generates new text based on the input it receives. It can create answers to questions, continue stories, or generate other forms of text.

**Retrieval-Augmented Generation** integrates these two systems. When the model is asked a question or given some context, it first uses the retrieval system to find relevant information. Then, it uses this information along with its own generation capabilities to produce a more detailed, coherent response than it could have done alone.

In simpler terms, imagine you're trying to answer a tricky quest

In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))

Saving 2604.26525v2.pdf to 2604.26525v2.pdf
Saving Futurewise_Synopsis_final.pdf to Futurewise_Synopsis_final (1).pdf
Saving MLBOOK.pdf to MLBOOK.pdf
Uploaded files: ['2604.26525v2.pdf', 'Futurewise_Synopsis_final (1).pdf', 'MLBOOK.pdf']


In [ ]:
import fitz  # PyMuPDF

documents = []

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        pdf = fitz.open(filename)

        for page_num, page in enumerate(pdf):

            text = page.get_text("text").strip()

            if text:
                documents.append({
                    "source": filename,
                    "page": page_num + 1,
                    "text": text
                })

        pdf.close()

print("Total pages with readable text:", len(documents))

for doc in documents:
    print(
        f"File: {doc['source']} | "
        f"Page: {doc['page']} | "
        f"Characters: {len(doc['text'])}"
    )

Total pages with readable text: 223
File: 2604.26525v2.pdf | Page: 1 | Characters: 5456
File: 2604.26525v2.pdf | Page: 2 | Characters: 5844
File: 2604.26525v2.pdf | Page: 3 | Characters: 6101
File: 2604.26525v2.pdf | Page: 4 | Characters: 5054
File: 2604.26525v2.pdf | Page: 5 | Characters: 4841
File: 2604.26525v2.pdf | Page: 6 | Characters: 5596
File: 2604.26525v2.pdf | Page: 7 | Characters: 5475
File: 2604.26525v2.pdf | Page: 8 | Characters: 6367
File: 2604.26525v2.pdf | Page: 9 | Characters: 5349
File: 2604.26525v2.pdf | Page: 10 | Characters: 5164
File: 2604.26525v2.pdf | Page: 11 | Characters: 5822
File: 2604.26525v2.pdf | Page: 12 | Characters: 6372
File: 2604.26525v2.pdf | Page: 13 | Characters: 9195
File: 2604.26525v2.pdf | Page: 14 | Characters: 5254
File: 2604.26525v2.pdf | Page: 15 | Characters: 5151
File: 2604.26525v2.pdf | Page: 16 | Characters: 4179
File: Futurewise_Synopsis_final (1).pdf | Page: 1 | Characters: 500
File: Futurewise_Synopsis_final (1).pdf | Page: 2 | Chara

In [ ]:
for doc in documents[:3]:
    print(f"\nSource: {doc['source']}")
    print(f"Page: {doc['page']}")
    print(doc["text"][:500])
    print("-" * 60)


Source: 2604.26525v2.pdf
Page: 1
1
PRAG: End-to-End Privacy-Preserving
Retrieval-Augmented Generation
Zhijun Li, Minghui Xu, Member, IEEE, Huayi Qi, Wenxuan Yu, Tingchuang Zhang, Qiao Zhang, GuangYong
Shang, Zhen Ma, Xiuzhen Cheng, Fellow, IEEE
This work has been submitted to the IEEE for possible
publication. Copyright may be transferred without notice, after
which this version may no longer be accessible.
Abstract—Retrieval-Augmented Generation (RAG) is essential
for enhancing Large Language Models (LLMs) with external
knowle
------------------------------------------------------------

Source: 2604.26525v2.pdf
Page: 2
2
mation leakage by injecting noise into documents [16]–[18].
While these approaches protect statistical privacy, they do not
provide provable data confidentiality; the cloud still stores per-
turbed document representations in plaintext and can observe
access patterns during retrieval. RemoteRAG [7] improves
upon this by perturbing user queries, preventing the cloud 

In [ ]:
print("Number of uploaded files:", len(uploaded))

for filename in uploaded.keys():
    print(filename)

Number of uploaded files: 3
2604.26525v2.pdf
Futurewise_Synopsis_final (1).pdf
MLBOOK.pdf


In [ ]:
from collections import Counter

page_counts = Counter(
    doc["source"] for doc in documents
)

print("PDFs successfully extracted:")

for filename in uploaded.keys():
    print(
        filename,
        "->",
        page_counts.get(filename, 0),
        "pages with readable text"
    )

PDFs successfully extracted:
2604.26525v2.pdf -> 16 pages with readable text
Futurewise_Synopsis_final (1).pdf -> 19 pages with readable text
MLBOOK.pdf -> 188 pages with readable text


##chunking pdfs

In [ ]:
def chunk_text(text, chunk_size=180, overlap=40):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks


chunks = []

for doc in documents:
    page_chunks = chunk_text(doc["text"])

    for chunk_id, chunk in enumerate(page_chunks):
        chunks.append({
            "source": doc["source"],
            "page": doc["page"],
            "chunk_id": chunk_id,
            "text": chunk
        })

print("Total chunks created:", len(chunks))

Total chunks created: 593


In [ ]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i + 1}")
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Text:", chunk["text"][:500])
    print("-" * 60)


Chunk 1
Source: 2604.26525v2.pdf
Page: 1
Chunk ID: 0
Text: 1 PRAG: End-to-End Privacy-Preserving Retrieval-Augmented Generation Zhijun Li, Minghui Xu, Member, IEEE, Huayi Qi, Wenxuan Yu, Tingchuang Zhang, Qiao Zhang, GuangYong Shang, Zhen Ma, Xiuzhen Cheng, Fellow, IEEE This work has been submitted to the IEEE for possible publication. Copyright may be transferred without notice, after which this version may no longer be accessible. Abstract—Retrieval-Augmented Generation (RAG) is essential for enhancing Large Language Models (LLMs) with external knowle
------------------------------------------------------------

Chunk 2
Source: 2604.26525v2.pdf
Page: 1
Chunk ID: 1
Text: active PRAG-II leverages client assistance to match the accuracy of non-private RAG. To ensure robust semantic ordering, we introduce Operation-Error Estimation (OEE), a mechanism that stabilizes ranking against homomorphic noise. Experiments on large-scale datasets demonstrate that PRAG achieves competitive recall (

Install and load the embedding model

In [ ]:
%pip install -q sentence-transformers faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBED_MODEL)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


Convert your chunks into vectors

In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]

if not chunk_texts:
    raise ValueError("No chunks found. Run the chunking step first.")

embeddings = embedder.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Number of chunks: 593
Embedding shape: (593, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index created successfully!")
print("Vectors stored:", index.ntotal)
print("Vector dimension:", index.d)

FAISS index created successfully!
Vectors stored: 593
Vector dimension: 384


In [ ]:
def retrieve_documents(query, top_k=3):

    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    k = min(top_k, index.ntotal)

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        item = chunks[int(idx)].copy()
        item["score"] = float(score)
        results.append(item)

    return results

TESTING MODEL works or not

In [ ]:
query = "What is the main objective of the Futurewise project?"

results = retrieve_documents(query, top_k=3)

for result in results:
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Similarity:", round(result["score"], 3))
    print("Text:", result["text"][:500])
    print("-" * 60)

Source: MLBOOK.pdf
Page: 161
Similarity: 0.385
Text: 152 CHAPTER 11. DELAYED-REINFORCEMENT LEARNING could be modiﬁed to have several places in the grid that give positive rewards. This possibility presents an interesting way to generalize the classical notion of a “goal” in AI planning systems—even in those that do no learning. Instead of representing a goal as a condition to be achieved, we represent a “goal struc- ture” as a set of rewards to be given for achieving various conditions. Then, the generalized “goal” becomes maximizing discounted fu
------------------------------------------------------------
Source: MLBOOK.pdf
Page: 176
Similarity: 0.371
Text: either succeeds or fails. After producing a plan, it analyzes its successful and its unsuccessful choices and attempts to explain them in terms of its domain theory. Using an EBL-like process, it is able to produce useful control rules such as:
------------------------------------------------------------
Source: Futurewise_Synopsi

In [ ]:
keywords = [
    "objective",
    "aim",
    "purpose",
    "problem statement",
    "proposed system",
    "project overview"
]

for doc in documents:
    if "Futurewise" not in doc["source"]:
        continue

    text_lower = doc["text"].lower()

    if any(keyword in text_lower for keyword in keywords):
        print(f"\nPage: {doc['page']}")
        print("Source:", doc["source"])
        print(doc["text"][:1200])
        print("-" * 70)


Page: 6
Source: Futurewise_Synopsis_final (1).pdf
2 
 
 
CONTENTS: 
 
 
Sr.No. 
Particulars 
Page Number 
1 
Introduction 
           03 
2 
Literature Survey 
           05 
3 
Problem Statement 
           07 
4 
Objectives 
           07 
5 
Methodology/Planning of work 
           08 
6 
Hardware/Software Requirement 
           10 
7 
Project Schedule (Timeline) 
           13 
8 
References 
           14 
 
 
 
 
 
 
Remarks by Project Guide :(Accepted/Rejected/Suggested for Corrections) 
 
 
 
 
 
 
 
 
Remarks by Project Panel: (Accepted/Rejected/Suggested for Corrections) 
 
 
 
 
 
 
 
 
 
 
 
 
 
Project Guide                        Project-III Coordinator                                  HOD CSE(AIML) 
Page 6 of 19 - Integrity Submission
Submission ID
trn:oid:::3117:613858174
Page 6 of 19 - Integrity Submission
Submission ID
trn:oid:::3117:613858174
----------------------------------------------------------------------

Page: 8
Source: Futurewise_Synopsis_final (1).pdf
4 

In [ ]:
query = (
    "Futurewise project's problem statement, "
    "aim, objectives, and proposed solution"
)

results = retrieve_documents(query, top_k=5)

for result in results:
    print("\nSource:", result["source"])
    print("Page:", result["page"])
    print("Similarity:", round(result["score"], 3))
    print(result["text"][:700])
    print("-" * 70)


Source: Futurewise_Synopsis_final (1).pdf
Page: 6
Similarity: 0.455
2 CONTENTS: Sr.No. Particulars Page Number 1 Introduction 03 2 Literature Survey 05 3 Problem Statement 07 4 Objectives 07 5 Methodology/Planning of work 08 6 Hardware/Software Requirement 10 7 Project Schedule (Timeline) 13 8 References 14 Remarks by Project Guide :(Accepted/Rejected/Suggested for Corrections) Remarks by Project Panel: (Accepted/Rejected/Suggested for Corrections) Project Guide Project-III Coordinator HOD CSE(AIML) Page 6 of 19 - Integrity Submission Submission ID trn:oid:::3117:613858174 Page 6 of 19 - Integrity Submission Submission ID trn:oid:::3117:613858174
----------------------------------------------------------------------

Source: Futurewise_Synopsis_final (1).pdf
Page: 1
Similarity: 0.404
D. Y. Patil College of Engineering and Technology, Kolhapur (Kasaba Bawada) Futurewise_Synopsis_final.docx Harsh chougale Document Details Submission ID trn:oid:::3117:613858174 Submission Date Aug 12, 20

In [ ]:
for doc in documents:
    if "Futurewise" not in doc["source"]:
        continue

    text = doc["text"]
    text_lower = text.lower()

    if (
        "objectives" in text_lower
        or "problem statement" in text_lower
    ):
        print("\n" + "=" * 70)
        print("PDF page:", doc["page"])
        print(text[:2500])


PDF page: 6
2 
 
 
CONTENTS: 
 
 
Sr.No. 
Particulars 
Page Number 
1 
Introduction 
           03 
2 
Literature Survey 
           05 
3 
Problem Statement 
           07 
4 
Objectives 
           07 
5 
Methodology/Planning of work 
           08 
6 
Hardware/Software Requirement 
           10 
7 
Project Schedule (Timeline) 
           13 
8 
References 
           14 
 
 
 
 
 
 
Remarks by Project Guide :(Accepted/Rejected/Suggested for Corrections) 
 
 
 
 
 
 
 
 
Remarks by Project Panel: (Accepted/Rejected/Suggested for Corrections) 
 
 
 
 
 
 
 
 
 
 
 
 
 
Project Guide                        Project-III Coordinator                                  HOD CSE(AIML) 
Page 6 of 19 - Integrity Submission
Submission ID
trn:oid:::3117:613858174
Page 6 of 19 - Integrity Submission
Submission ID
trn:oid:::3117:613858174

PDF page: 11
7 
 
3. Problem Statement and Objectives 
3.1 Problem Statement 
Retail equity traders operate at a persistent disadvantage because global financial

In [ ]:
def retrieve_from_source(query, source_name, top_k=5):

    # Find chunks belonging to the selected PDF
    candidate_indices = [
        i for i, chunk in enumerate(chunks)
        if source_name.lower() in chunk["source"].lower()
    ]

    if not candidate_indices:
        print("No chunks found for:", source_name)
        return []

    # Embed the question
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    # Compare the query with the selected PDF's chunks
    candidate_embeddings = embeddings[candidate_indices]

    scores = candidate_embeddings @ query_embedding

    # Rank chunks by similarity
    ranked_positions = np.argsort(scores)[::-1][:top_k]

    results = []

    for position in ranked_positions:
        idx = candidate_indices[int(position)]

        item = chunks[idx].copy()
        item["score"] = float(scores[position])

        results.append(item)

    return results

In [ ]:
query = (
    "Futurewise section 3.2 Objectives: "
    "automated financial news intelligence pipeline, "
    "real-time news aggregation, named entity recognition, "
    "FinBERT sentiment classification, portfolio correlation, "
    "and personalized risk indicators"
)

results = retrieve_from_source(
    query=query,
    source_name="Futurewise_Synopsis_final (1).pdf",
    top_k=5
)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Page:", result["page"])
    print("Similarity:", round(result["score"], 3))
    print(result["text"][:800])
    print("-" * 60)


Result 1
Page: 11
Similarity: 0.737
decision-making, frequently missing opportunities to mitigate risk and absorbing unmanaged losses during sudden, sentiment-driven market swings. 3.2 Objectives The primary goal of the Futurewise project is to build an automated, end-to-end intelligence pipeline that transforms unstructured financial news into clear, portfolio-specific risk indicators. To achieve this, the project is structured around seven core technical objectives: 1. Real-Time News Aggregation: Build a scalable data-ingestion pipeline that continuously pulls unstructured financial text from external news APIs (such as NewsAPI and GNews) as well as localized financial RSS feeds. 2. Precision Entity Extraction and Resolution: Implement a Natural Language Processing (NER) module to identify corporate entities and ticker sym
------------------------------------------------------------

Result 2
Page: 11
Similarity: 0.708
7 3. Problem Statement and Objectives 3.1 Problem Statement Reta

In [ ]:
################################################################

In [ ]:
print("Model loaded:", "model" in globals())
print("Tokenizer loaded:", "tokenizer" in globals())
print("Generation function available:", "generate_answer" in globals())

Model loaded: True
Tokenizer loaded: True
Generation function available: True


In [ ]:
def rag_answer(question, source_name, top_k=3):

    # 1. Retrieve relevant passages
    retrieved = retrieve_from_source(
        query=question,
        source_name=source_name,
        top_k=top_k
    )

    if not retrieved:
        return {
            "answer": "No relevant passages were found.",
            "sources": []
        }

    # 2. Prepare the context
    context_parts = []

    for i, item in enumerate(retrieved, start=1):
        context_parts.append(
            f"[Source {i}: {item['source']}, "
            f"PDF page {item['page']}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_parts)

    # 3. Ask the LLM to answer from the context
    prompt = f"""
You are a helpful AI assistant answering questions
using the user's uploaded documents.

Rules:
- Answer using the supplied context.
- Do not invent facts.
- If the context does not answer the question,
  say that the information was not found.
- Cite supporting passages using [Source 1],
  [Source 2], etc.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = generate_answer(
        prompt,
        max_new_tokens=300
    )

    # 4. Return the answer and source metadata
    sources = [
        {
            "source": item["source"],
            "page": item["page"],
            "similarity": round(item["score"], 3)
        }
        for item in retrieved
    ]

    return {
        "answer": answer,
        "sources": sources
    }

In [ ]:
response = rag_answer(
    question=(
        "What are the main objectives of the Futurewise project?"
    ),
    source_name="Futurewise_Synopsis_final (1).pdf",
    top_k=3
)

print("ANSWER:\n")
print(response["answer"])

print("\nSOURCES:\n")

for source in response["sources"]:
    print(
        f"{source['source']} | "
        f"Page {source['page']} | "
        f"Similarity {source['similarity']}"
    )

ANSWER:

Based on the information provided in the context, there are no specific main objectives listed directly. The context focuses primarily on the project schedule and methodology rather than detailing the objectives. Therefore, the information was not found regarding the main objectives of the Futurewise project.

SOURCES:

Futurewise_Synopsis_final (1).pdf | Page 17 | Similarity 0.385
Futurewise_Synopsis_final (1).pdf | Page 12 | Similarity 0.369
Futurewise_Synopsis_final (1).pdf | Page 6 | Similarity 0.353


In [ ]:
def rag_answer(question, source_name, top_k=5):

    # Expand the search query to improve retrieval
    retrieval_query = (
        question
        + " Futurewise section 3.2 Objectives "
        + "primary goal automated financial news intelligence pipeline "
        + "real-time news aggregation entity extraction NER "
        + "FinBERT sentiment classification portfolio correlation "
        + "personalized portfolio risk indicators"
    )

    # 1. Retrieve relevant passages
    retrieved = retrieve_from_source(
        query=retrieval_query,
        source_name=source_name,
        top_k=top_k
    )

    if not retrieved:
        return {
            "answer": "No relevant passages were found.",
            "sources": []
        }

    # 2. Prepare the context
    context_parts = []

    for i, item in enumerate(retrieved, start=1):
        context_parts.append(
            f"[Source {i}: {item['source']}, "
            f"PDF page {item['page']}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_parts)

    # 3. Generate an answer from the retrieved context
    prompt = f"""
You are a helpful AI assistant.

Answer the question using the supplied document context.

Instructions:
- Answer the question directly.
- Include important details supported by the context.
- Do not invent facts.
- If the answer is unavailable, say so.
- Cite supporting passages as [Source 1], [Source 2], etc.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = generate_answer(
        prompt,
        max_new_tokens=350
    )

    sources = [
        {
            "source": item["source"],
            "page": item["page"],
            "similarity": round(item["score"], 3)
        }
        for item in retrieved
    ]

    return {
        "answer": answer,
        "sources": sources
    }

In [ ]:
response = rag_answer(
    question="What are the main objectives of the Futurewise project?",
    source_name="Futurewise_Synopsis_final (1).pdf",
    top_k=5
)

print("ANSWER:\n")
print(response["answer"])

print("\nSOURCES:\n")

for source in response["sources"]:
    print(
        f"{source['source']} | "
        f"Page {source['page']} | "
        f"Similarity: {source['similarity']}"
    )

ANSWER:

The main objectives of the Futurewise project are to build an automated, end-to-end intelligence pipeline that transforms unstructured financial news into clear, portfolio-specific risk indicators. Specifically, these objectives include:

1. **Real-Time News Aggregation**: Building a scalable data-injection pipeline that continuously pulls unstructured financial text from external news APIs and localized financial RSS feeds.
   
2. **Precision Entity Extraction and Resolution**: Implementing a Natural Language Processing (NER) module to identify corporate entities and ticker symbols within raw text, using a custom alias-mapping dictionary to resolve varying names.

3. **Contextual Sentiment Classification**: Deploying a domain-specific bidirectional transformer model, FinBERT, to evaluate extracted sentences and assign accurate bullish, bearish, or neutral sentiment polarity scores.

4. **Dynamic Portfolio Correlation**: Engineering a matching engine that cross-references extr

In [ ]:
def retrieve_all_documents(query, top_k=5):

    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    scores = embeddings @ query_embedding

    # Rank all chunks across all uploaded PDFs
    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in ranked_indices:
        item = chunks[int(idx)].copy()
        item["score"] = float(scores[idx])
        results.append(item)

    return results


In [ ]:
test_questions = [
    "What are the objectives of the Futurewise project?",
    "Explain the machine learning concepts discussed in MLBOOK.",
    "What does the research paper say about retrieval-augmented generation?"
]

for question in test_questions:

    print("\nQUESTION:", question)

    results = retrieve_all_documents(
        question,
        top_k=3
    )

    for result in results:
        print(
            result["source"],
            "| Page:", result["page"],
            "| Similarity:", round(result["score"], 3)
        )


QUESTION: What are the objectives of the Futurewise project?
MLBOOK.pdf | Page: 161 | Similarity: 0.417
Futurewise_Synopsis_final (1).pdf | Page: 17 | Similarity: 0.386
MLBOOK.pdf | Page: 176 | Similarity: 0.383

QUESTION: Explain the machine learning concepts discussed in MLBOOK.
MLBOOK.pdf | Page: 20 | Similarity: 0.597
MLBOOK.pdf | Page: 22 | Similarity: 0.548
MLBOOK.pdf | Page: 11 | Similarity: 0.546

QUESTION: What does the research paper say about retrieval-augmented generation?
2604.26525v2.pdf | Page: 13 | Similarity: 0.497
2604.26525v2.pdf | Page: 12 | Similarity: 0.479
MLBOOK.pdf | Page: 7 | Similarity: 0.423


In [ ]:
def smart_retrieve(query, top_k=5):

    q = query.lower()

    # Select a document when the question identifies it
    if "futurewise" in q:
        source_filter = "Futurewise_Synopsis_final"
    elif "mlbook" in q:
        source_filter = "MLBOOK.pdf"
    elif any(term in q for term in [
        "research paper",
        "2604.26525",
        "retrieval-augmented generation",
        "rag"
    ]):
        source_filter = "2604.26525v2.pdf"
    else:
        source_filter = None

    # Use the existing retrieval function
    if source_filter:
        results = retrieve_from_source(
            query=query,
            source_name=source_filter,
            top_k=top_k
        )
    else:
        results = retrieve_all_documents(
            query=query,
            top_k=top_k
        )

    return results

In [ ]:
question = "What are the objectives of the Futurewise project?"

results = smart_retrieve(question, top_k=3)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Similarity:", round(result["score"], 3))
    print(result["text"][:500])
    print("-" * 60)



Result 1
Source: Futurewise_Synopsis_final (1).pdf
Page: 17
Similarity: 0.386
13 6. Project Schedule Phase Activity Duration Month 1 Requirement Analysis & Literature Survey 2 Weeks July 2025 2 System Design 2 Weeks August 2025 3 News API Integration & Data Collection 3 Weeks Aug–Sep 2025 4 NLP Preprocessing & NER 3 Weeks September 2025 5 FinBERT Sentiment Analysis 3 Weeks October 2025 6 Portfolio Mapping & Risk Score 3 Weeks Oct–Nov 2025 7 Frontend Dashboard 4 Weeks November 2025 8 Alert System Integration 2 Weeks December 2025 9 Testing & Optimization 3 Weeks January 20
------------------------------------------------------------

Result 2
Source: Futurewise_Synopsis_final (1).pdf
Page: 12
Similarity: 0.361
8 notifications whenever a user's holdings experience significant positive or negative momentum. 7. Interactive Dashboard and Decision Support: Design a responsive web application that displays an overall Portfolio Health Score, stock-by-stock sentiment charts, and historical ris

In [ ]:
import re
import numpy as np

def retrieve_objectives(query, top_k=5):

    source_name = "Futurewise_Synopsis_final"

    # Select chunks from the correct document
    candidate_indices = [
        i for i, chunk in enumerate(chunks)
        if source_name.lower() in chunk["source"].lower()
    ]

    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    # Semantic similarity
    candidate_vectors = embeddings[candidate_indices]
    semantic_scores = candidate_vectors @ query_embedding

    # Keyword relevance
    keywords = [
        "objectives",
        "primary goal",
        "real-time news aggregation",
        "entity extraction",
        "finbert",
        "sentiment classification",
        "portfolio correlation",
        "risk indicators",
        "interactive dashboard"
    ]

    ranked = []

    for position, idx in enumerate(candidate_indices):
        text = chunks[idx]["text"].lower()

        keyword_hits = sum(
            1 for keyword in keywords
            if keyword in text
        )

        # Combine semantic and keyword scores
        score = (
            0.60 * float(semantic_scores[position])
            + 0.40 * (keyword_hits / len(keywords))
        )

        ranked.append((score, idx))

    ranked.sort(reverse=True)

    results = []

    for score, idx in ranked[:top_k]:
        item = chunks[idx].copy()
        item["score"] = score
        results.append(item)

    return results


In [ ]:
query = (
    "What are the seven main objectives of the Futurewise project?"
)

results = retrieve_objectives(query, top_k=5)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Combined score:", round(result["score"], 3))
    print(result["text"][:900])
    print("-" * 60)


Result 1
Source: Futurewise_Synopsis_final (1).pdf
Page: 11
Combined score: 0.568
decision-making, frequently missing opportunities to mitigate risk and absorbing unmanaged losses during sudden, sentiment-driven market swings. 3.2 Objectives The primary goal of the Futurewise project is to build an automated, end-to-end intelligence pipeline that transforms unstructured financial news into clear, portfolio-specific risk indicators. To achieve this, the project is structured around seven core technical objectives: 1. Real-Time News Aggregation: Build a scalable data-ingestion pipeline that continuously pulls unstructured financial text from external news APIs (such as NewsAPI and GNews) as well as localized financial RSS feeds. 2. Precision Entity Extraction and Resolution: Implement a Natural Language Processing (NER) module to identify corporate entities and ticker symbols within raw text, using a custom alias- mapping dictionary to resolve varying names (for example
----------------

In [ ]:
query = (
    "Futurewise section 3.2 Objectives: all seven core technical objectives, "
    "including news aggregation, entity extraction, sentiment classification, "
    "portfolio correlation, risk indicators, alerts, and dashboard"
)

results = retrieve_objectives(query, top_k=5)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} | PDF page {result['page']} ---")
    print(result["text"])
    print("-" * 80)


--- Result 1 | PDF page 11 ---
decision-making, frequently missing opportunities to mitigate risk and absorbing unmanaged losses during sudden, sentiment-driven market swings. 3.2 Objectives The primary goal of the Futurewise project is to build an automated, end-to-end intelligence pipeline that transforms unstructured financial news into clear, portfolio-specific risk indicators. To achieve this, the project is structured around seven core technical objectives: 1. Real-Time News Aggregation: Build a scalable data-ingestion pipeline that continuously pulls unstructured financial text from external news APIs (such as NewsAPI and GNews) as well as localized financial RSS feeds. 2. Precision Entity Extraction and Resolution: Implement a Natural Language Processing (NER) module to identify corporate entities and ticker symbols within raw text, using a custom alias- mapping dictionary to resolve varying names (for example, mapping "Alphabet", "Google", and "GOOGL" to one unified database 

In [ ]:
source_name = "Futurewise_Synopsis_final"

objective_chunks = [
    chunk for chunk in chunks
    if source_name.lower() in chunk["source"].lower()
    and chunk["page"] in [11, 12]
]

objective_chunks.sort(key=lambda chunk: (chunk["page"], chunk["chunk_id"]))

for chunk in objective_chunks:
    print(f"\n--- PDF page {chunk['page']}, chunk {chunk['chunk_id']} ---")
    print(chunk["text"])


--- PDF page 11, chunk 0 ---
7 3. Problem Statement and Objectives 3.1 Problem Statement Retail equity traders operate at a persistent disadvantage because global financial news is voluminous, fast-moving, and entirely unstructured. Every trading day generates thousands of corporate earnings updates, macroeconomic reports, regulatory filings, and market narratives across distributed media channels. Today, an individual investor has no automated tool to filter this broad stream of unstructured data down to the specific events that affect their personal stock holdings. Even when an investor successfully identifies a relevant news article, they lack the computational tools to objectively calculate how much that headline should alter their portfolio's risk profile. Existing platforms either display broad, market-wide sentiment—which ignores how a user's specific assets are allocated—or rely on delayed, subjective ratings from human analysts. This creates severe information overload and in

In [ ]:
# Collect chunks containing the objectives section
objective_text = "\n".join(
    chunk["text"]
    for chunk in objective_chunks
    if chunk["page"] in [11, 12]
)

prompt = f"""
You are a document-based AI assistant.

Answer the question using only the supplied document text.
List all seven objectives, numbered 1 to 7.
Explain each objective briefly and accurately.
Do not add objectives that are not stated in the document.

Document text:
{objective_text}

Question:
What are the seven main objectives of the Futurewise project?

Answer:
"""

answer = generate_answer(prompt, max_new_tokens=500)
print(answer)

The seven main objectives of the Futurewise project are:

1. **Real-Time News Aggregation**: Build a scalable data-ingestion pipeline that continuously pulls unstructured financial text from external news APIs and localized financial RSS feeds.
2. **Precision Entity Extraction and Resolution**: Implement a Natural Language Processing (NER) module to identify corporate entities and ticker symbols within raw text, using a custom alias-mapping dictionary to resolve varying names.
3. **Contextual Sentiment Classification**: Deploy FinBERT, a domain-specific bidirectional transformer model fine-tuned on financial text, to evaluate extracted sentences and assign accurate bullish, bearish, or neutral sentiment polarity scores.
4. **Dynamic Portfolio Correlation**: Engineer a matching engine that cross-references extracted corporate entities and their sentiment scores directly against the user’s uploaded portfolio holdings across rolling time windows.
5. **Algorithmic Risk and Upside Scoring**

In [ ]:
def rag_answer(question, top_k=5):
    q = question.lower()

    # Route to a specific PDF when the question names one.
    if "futurewise" in q:
        results = retrieve_objectives(question, top_k=top_k)
    elif "mlbook" in q:
        results = retrieve_from_source(
            question, "MLBOOK.pdf", top_k=top_k
        )
    elif any(term in q for term in [
        "research paper", "2604.26525", "retrieval-augmented generation"
    ]):
        results = retrieve_from_source(
            question, "2604.26525v2.pdf", top_k=top_k
        )
    else:
        results = retrieve_all_documents(question, top_k=top_k)

    if not results:
        return {
            "answer": "I couldn't find relevant passages in the documents.",
            "sources": []
        }

    context = "\n\n".join(
        f"[Source {i}: {item['source']}, PDF page {item['page']}]\n"
        f"{item['text']}"
        for i, item in enumerate(results, start=1)
    )

    prompt = f"""
You are a document-grounded AI assistant.

Instructions:
- Answer using only the supplied context.
- Do not invent facts or details.
- If the context does not answer the question, say so.
- Cite supporting passages as [Source 1], [Source 2], etc.
- Give a clear, organized answer.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = generate_answer(prompt, max_new_tokens=400)

    sources = [
        {
            "source": item["source"],
            "page": item["page"],
            "similarity": round(item["score"], 3)
        }
        for item in results
    ]

    return {"answer": answer, "sources": sources}

In [ ]:
result = rag_answer(
    "What are the seven main objectives of the Futurewise project?"
)

print("ANSWER:\n")
print(result["answer"])

print("\nSOURCES:\n")
for source in result["sources"]:
    print(source)

ANSWER:

The seven main objectives of the Futurewise project are:

1. Real-Time News Aggregation
2. Precision Entity Extraction and Resolution
3. Contextual Sentiment Classification
4. Dynamic Portfolio Correlation
5. Interactive Dashboard and Decision Support
6. 8 Notifications Whenever a User's Holdings Experience Significant Positive or Negative Momentum
7. Project Schedule Planning

SOURCES:

{'source': 'Futurewise_Synopsis_final (1).pdf', 'page': 11, 'similarity': 0.568}
{'source': 'Futurewise_Synopsis_final (1).pdf', 'page': 12, 'similarity': 0.319}
{'source': 'Futurewise_Synopsis_final (1).pdf', 'page': 17, 'similarity': 0.293}
{'source': 'Futurewise_Synopsis_final (1).pdf', 'page': 6, 'similarity': 0.272}
{'source': 'Futurewise_Synopsis_final (1).pdf', 'page': 11, 'similarity': 0.268}


In [ ]:
# Keep only chunks from the verified objectives pages.
objective_chunks = [
    chunk for chunk in chunks
    if "Futurewise_Synopsis_final" in chunk["source"]
    and chunk["page"] in [11, 12]
]

objective_chunks.sort(
    key=lambda chunk: (chunk["page"], chunk["chunk_id"])
)

# Use chunk 1 and chunk 2 on page 11, plus chunk 0 on page 12.
# These contain the objective list and avoid unrelated schedule pages.
selected_chunks = [
    chunk for chunk in objective_chunks
    if (chunk["page"] == 11 and chunk["chunk_id"] in [1, 2])
    or (chunk["page"] == 12 and chunk["chunk_id"] == 0)
]

context = "\n\n".join(chunk["text"] for chunk in selected_chunks)

prompt = f"""
Answer using only the document context below.

Question: What are the seven main objectives of the Futurewise project?

Rules:
- List exactly seven objectives.
- Preserve their original names and order.
- Do not treat continuation text as a separate objective.
- Do not include the project schedule.
- Briefly explain each objective.
- If any objective cannot be identified, say so instead of guessing.

Document context:
{context}

Answer:
"""

answer = generate_answer(prompt, max_new_tokens=500)
print(answer)

The seven main objectives of the Futurewise project are:

1. **Real-Time News Aggregation**: Build a scalable data-ingestion pipeline that continuously pulls unstructured financial text from external news APIs and localized financial RSS feeds.
   
2. **Precision Entity Extraction and Resolution**: Implement a Natural Language Processing (NER) module to identify corporate entities and ticker symbols within raw text, using a custom alias-mapping dictionary to resolve varying names.

3. **Contextual Sentiment Classification**: Deploy a domain-specific bidirectional transformer model, FinBERT, to evaluate extracted sentences and assign accurate bullish, bearish, or neutral sentiment polarity scores.

4. **Dynamic Portfolio Correlation**: Engineer a matching engine that cross-references extracted corporate entities and their sentiment scores directly against the user’s uploaded portfolio holdings across rolling time windows.

5. **Algorithmic Risk and Upside Scoring**: Formulate a mathemat

In [ ]:
question = (
    "What does the research paper say about "
    "retrieval-augmented generation?"
)

results = retrieve_from_source(
    query=question,
    source_name="2604.26525v2.pdf",
    top_k=3
)

for i, item in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("PDF:", item["source"])
    print("Page:", item["page"])
    print("Similarity:", round(item["score"], 3))
    print(item["text"])
    print("-" * 80)


--- Result 1 ---
PDF: 2604.26525v2.pdf
Page: 13
Similarity: 0.497
Huang, S. Gupta, Z. Zhong, K. Li, and D. Chen, “Privacy implications of retrieval-based language models,” in 2023 Conference on Empirical Methods in Natural Language Processing, EMNLP 2023. Association for Computational Linguistics (ACL), 2023, pp. 14 887–14 902. [37] S. Zeng, J. Zhang, P. He, Y. Xing, Y. Liu, H. Xu, J. Ren, S. Wang, D. Yin, Y. Chang et al., “The good and the bad: Exploring privacy issues in retrieval-augmented generation (rag),” arXiv preprint arXiv:2402.16893, 2024. [38] J. He, C. Liu, G. Hou, W. Jiang, and J. Li, “Press: Defending privacy in retrieval-augmented generation via embedding space shifting,” in ICASSP 2025-2025 IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP). IEEE, 2025, pp. 1–5. [39] D. Zhao, “Frag: Toward federated vector database management for collaborative and secure retrieval-augmented generation,” arXiv preprint arXiv:2410.13272, 2024. [40] T. E And

In [ ]:
# Find occurrences of RAG-related terms in the paper's extracted text.
terms = [
    "retrieval-augmented generation",
    "retrieval augmented generation",
    "rag",
    "retrieval-based language models"
]

for doc in documents:
    if "2604.26525v2.pdf" not in doc["source"]:
        continue

    text_lower = doc["text"].lower()
    matches = []

    for term in terms:
        start = 0
        while True:
            position = text_lower.find(term, start)
            if position == -1:
                break

            snippet_start = max(0, position - 250)
            snippet_end = min(len(doc["text"]), position + 450)

            matches.append(
                (position, term, doc["text"][snippet_start:snippet_end])
            )
            start = position + len(term)

    if matches:
        print(f"\n{'=' * 70}")
        print(f"PDF page: {doc['page']}")

        for _, term, snippet in sorted(matches):
            print(f"\nMatched term: {term}")
            print(snippet)


PDF page: 1

Matched term: rag
1
PRAG: End-to-End Privacy-Preserving
Retrieval-Augmented Generation
Zhijun Li, Minghui Xu, Member, IEEE, Huayi Qi, Wenxuan Yu, Tingchuang Zhang, Qiao Zhang, GuangYong
Shang, Zhen Ma, Xiuzhen Cheng, Fellow, IEEE
This work has been submitted to the IEEE for possible
publication. Copyright may be transferred without notice, after
which this version may no longer be accessible.
Abstract—Retrieval-Augmented Generation (RAG) is essential
for enhancing La

Matched term: retrieval-augmented generation
1
PRAG: End-to-End Privacy-Preserving
Retrieval-Augmented Generation
Zhijun Li, Minghui Xu, Member, IEEE, Huayi Qi, Wenxuan Yu, Tingchuang Zhang, Qiao Zhang, GuangYong
Shang, Zhen Ma, Xiuzhen Cheng, Fellow, IEEE
This work has been submitted to the IEEE for possible
publication. Copyright may be transferred without notice, after
which this version may no longer be accessible.
Abstract—Retrieval-Augmented Generation (RAG) is essential
for enhancing Large Language Mo